# Data and sample characteristics

Generate **Table S1** from the included metadata for **148 children**.
Select **Python (dbh)**, then run the cells in order. This notebook calls
`export_table_s1.py` to save the table and sex-ratio test under the repository’s `results/SI/`,
and displays both results below.
No signal recordings or completed preprocessing are needed.

After the [environment setup](../README.md#setup), open this notebook from the
repository root:

```bash
python -m jupyterlab 00_data/run_sample_characteristics.ipynb
```

See [README.md](README.md) for the shared data layout and input requirements.

## 1. Locate the files and check Python

The input is `participants.csv` beside this notebook. Outputs go to the repository’s `results/SI/`.
Paths are located from the repository, so no machine-specific path is needed.
The display uses the same 16-px style as the preprocessing notebook.

In [3]:
from pathlib import Path
import sys

import pandas as pd
import scipy
from IPython.display import HTML, Markdown, display

ROOT = next(
    (p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "00_data/export_table_s1.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")

DATA_DIR = ROOT / "00_data"
INFO_FILE = DATA_DIR / "participants.csv"
SCRIPT = DATA_DIR / "export_table_s1.py"
RESULT_DIR = ROOT / "results/SI"
STYLE_FILE = ROOT / "common/notebook_style.css"
display(HTML('<style id="dbh-notebook-style">' +
             STYLE_FILE.read_text(encoding="utf-8") + "</style>"))

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| SciPy:", scipy.__version__)
print("Participant table:", INFO_FILE.relative_to(ROOT))
print("Results:", RESULT_DIR.relative_to(ROOT))


Python: 3.12.7
pandas: 2.3.3 | SciPy: 1.16.3
Participant table: 00_data/participants.csv
Results: results/SI


## 2. Check the participant table

Require unique, nonblank IDs, sex coded `M`/`F`, and valid age values for all
148 children. The three study groups contain 50, 49 and 49 children.
Groups use completed years in `Age_group`; age summaries use `Age_month`.
These checks read the table without changing it.

In [5]:
info = pd.read_csv(INFO_FILE, encoding="utf-8-sig", dtype={"ID": str})
required = {"ID", "Gender", "Age_month", "Age_group"}
missing = required - set(info.columns)
if missing:
    raise ValueError(f"Missing participant columns: {sorted(missing)}")
if len(info) != 148:
    raise ValueError(f"Expected 148 participants; found {len(info)}.")
if info.ID.isna().any() or info.ID.str.strip().eq("").any() or info.ID.duplicated().any():
    raise ValueError("Participant IDs must be nonblank and unique.")
if not info.Gender.isin(["M", "F"]).all():
    raise ValueError("Gender must be coded M or F for every participant.")
for column in ["Age_group", "Age_month"]:
    info[column] = pd.to_numeric(info[column], errors="raise")
if not info.Age_group.isin([5, 6, 7, 8, 9, 10]).all():
    raise ValueError("Age_group must contain completed years 5 through 10.")
if not info.Age_month.between(60, 132, inclusive="left").all():
    raise ValueError("Age_month must be finite and between 60 and less than 132 months.")

groups = pd.cut(info.Age_group, [4, 6, 8, 10], labels=["5-6", "7-8", "9-10"])
counts = groups.value_counts(sort=False)
if counts.tolist() != [50, 49, 49]:
    raise ValueError(f"Expected study group sizes 50/49/49; found {counts.tolist()}.")
print("Participant checks passed.")
display(counts.rename("n").to_frame().rename_axis("Age group"))


Participant checks passed.


,n
Age group,
5-6,50
7-8,49
9-10,49


## 3. Generate and display Table S1

The table and sex-ratio test appear below and are saved as
`results/SI/Table_S1_sample_characteristics.csv` and `results/SI/Table_S1_note.txt`
at the repository root. Rerunning this cell replaces these two files.


In [7]:
from runpy import run_path

export_table_s1 = run_path(str(SCRIPT))["export_table_s1"]
table, note = export_table_s1(INFO_FILE, RESULT_DIR)
display(Markdown("**Table S1. Sample characteristics by age group**"))
display(table.style.hide(axis="index").set_uuid("table_s1"))
display(Markdown(note.strip()))
print("Saved:", RESULT_DIR.relative_to(ROOT) / "Table_S1_sample_characteristics.csv")
print("Saved:", RESULT_DIR.relative_to(ROOT) / "Table_S1_note.txt")


**Table S1. Sample characteristics by age group**

Age group,n,Boys/Girls,"Age (months), M ± SD","Age (months), range","Age (years), M ± SD"
5-6,50,29/21,73.0 ± 7.3,61–83,6.08 ± 0.61
7-8,49,26/23,92.8 ± 6.9,84–107,7.73 ± 0.58
9-10,49,22/27,119.1 ± 6.7,108–131,9.92 ± 0.56
All,148,77/71,94.8 ± 20.2,61–131,7.90 ± 1.68


Sex ratio by age group: chi2(2) = 1.73, p = 0.420

Saved: results/SI/Table_S1_sample_characteristics.csv
Saved: results/SI/Table_S1_note.txt


For signal processing, continue in
[01_preprocessing/run_preprocessing.ipynb](../01_preprocessing/run_preprocessing.ipynb).